# Comparing the candidate recovery metrics

Each per-metric notebook answers "can this be measured here"; this one answers "which of them
should the project commit to". It reads the scorecard rows the other notebooks wrote, then asks
the question no single-metric notebook can: whether the survivors measure the same thing or
different things.

Run the per-metric notebooks first. The decision this produces is one primary outcome plus two or
three pre-declared secondaries, written down before any model is fitted, so that reporting the
best-performing outcome after the fact is not an option available later.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("metric_comparison")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

# The steps ratio below is built here rather than read from the steps notebook, so its
# windows have to be interpolated from the shared definition, never retyped: a literal
# copy silently stops matching the moment DEFAULT_WINDOWS changes, and the scorecard
# would then compare metrics computed over different windows.
BASE_LO, BASE_HI = WINDOWS["baseline"]
POST_LO, POST_HI = WINDOWS["postop"]

METRIC_CTE = srp_common.index_visit_cte(cfg) + f""",
    activity AS (
        SELECT
            a.person_id,
            DATE_DIFF(a.date, i.index_date, DAY) AS rel_day,
            a.steps
        FROM `{CDR}`.activity_summary AS a
        JOIN index_surgery AS i ON i.person_id = a.person_id
        WHERE a.steps IS NOT NULL
    ),
    steps_windows AS (
        SELECT
            person_id,
            APPROX_QUANTILES(IF(rel_day BETWEEN {BASE_LO} AND {BASE_HI}, steps, NULL),
                             100)[OFFSET(50)] AS baseline_steps,
            APPROX_QUANTILES(IF(rel_day BETWEEN {POST_LO} AND {POST_HI}, steps, NULL),
                             100)[OFFSET(50)] AS postop_steps,
            COUNTIF(rel_day BETWEEN {BASE_LO} AND {BASE_HI}) AS baseline_days,
            COUNTIF(rel_day BETWEEN {POST_LO} AND {POST_HI}) AS postop_days
        FROM activity
        GROUP BY person_id
    ),
    readmission AS (
        SELECT
            iv.person_id,
            MIN(DATE_DIFF(vo.visit_start_date, iv.visit_end_date, DAY)) AS days_to_readmission
        FROM index_visit AS iv
        JOIN `{CDR}`.visit_occurrence AS vo
          ON vo.person_id = iv.person_id
         AND vo.visit_occurrence_id != iv.visit_occurrence_id
         AND vo.visit_concept_id IN (9201, 262)
         AND vo.visit_start_date > iv.visit_end_date
        WHERE iv.rn = 1
        GROUP BY iv.person_id
    ),
    followup AS (
        SELECT
            iv.person_id,
            DATE_DIFF(MAX(op.observation_period_end_date), iv.visit_end_date, DAY)
                AS followup_days
        FROM index_visit AS iv
        JOIN `{CDR}`.observation_period AS op ON op.person_id = iv.person_id
        WHERE iv.rn = 1
        GROUP BY iv.person_id
    ),
    metrics AS (
        SELECT
            iv.person_id,
            iv.postop_los_days,
            -- NULL, not 0, for anyone observed less than the horizon: they are censored,
            -- not event-free, and scoring them 0 would bias the correlation.
            IF(f.followup_days >= 30,
               IF(r.days_to_readmission BETWEEN 0 AND 30, 1, 0),
               NULL) AS readmitted_30,
            IF(s.baseline_days >= 7 AND s.postop_days >= 7,
               SAFE_DIVIDE(s.postop_steps, NULLIF(s.baseline_steps, 0)),
               NULL) AS steps_recovery_ratio
        FROM index_visit AS iv
        LEFT JOIN steps_windows AS s ON s.person_id = iv.person_id
        LEFT JOIN readmission   AS r ON r.person_id = iv.person_id
        LEFT JOIN followup      AS f ON f.person_id = iv.person_id
        WHERE iv.rn = 1
    )"""

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "activity_summary": ["person_id", "date", "steps"],
    "observation_period": ["person_id", "observation_period_end_date"],
    "visit_occurrence": ["person_id", "visit_occurrence_id", "visit_concept_id", "visit_start_date", "visit_end_date"],
})

# 2-scorecard

### What did each per-metric notebook conclude?

The rows are read from the shared task cache and refuse to combine if they were produced against
different CDR releases or cohort builds. Coverage share is the fraction of the whole cohort for
which the metric is actually computable, not the fraction of those who have the source data.

In [ ]:
scorecard = srp_common.read_scorecards(cfg)

columns = ["metric", "family", "outcome_type", "cohort_patients", "patients_with_metric",
           "patients_analysable", "coverage_share", "horizon_days",
           "informative_missingness", "leakage_risk"]
present = [c for c in columns if c in scorecard.columns]

print(scorecard[present].to_string(index=False))
print()

# days_alive_out_of_hospital and discharge_disposition are in not_using/ and are
# deliberately absent: DAOH rests on death records this CDR does not usefully carry,
# and a discharge destination is recorded for about 5% of the cohort.
LIVE_METRICS = {"fitbit_steps", "fitbit_heart_rate", "fitbit_sleep", "length_of_stay",
                "readmission", "survey_physical_health"}
missing = LIVE_METRICS - set(scorecard["metric"])
if missing:
    print("not yet run: " + ", ".join(sorted(missing)))
retired = set(scorecard["metric"]) - LIVE_METRICS
if retired:
    print("stale scorecards from retired metrics, delete from _cache/: "
          + ", ".join(sorted(retired)))

### Which metrics survive on coverage alone?

Coverage is the first filter because no amount of clinical face validity rescues a metric that
exists for a tenth of the cohort. The bar is drawn here rather than argued: metrics are ranked,
and the ranking is what the rest of the notebook works from.

In [ ]:
COVERAGE_FLOOR = 0.50

ranked = scorecard.sort_values("coverage_share", ascending=False).reset_index(drop=True)
ranked["verdict"] = [
    "good" if s >= COVERAGE_FLOOR else ("warning" if s >= COVERAGE_FLOOR / 2 else "bad")
    for s in ranked["coverage_share"]
]

for _, row in ranked.iterrows():
    print(f"{srp_common.verdict_label(row['verdict']):<9} "
          f"{row['metric']:<28} {row['coverage_share']:6.1%}  "
          f"({int(row['patients_analysable']):,} of {int(row['cohort_patients']):,})")

fig, ax = plt.subplots(figsize=(10, 0.45 * len(ranked) + 2.2), facecolor=srp_common.SURFACE)
srp_common.style_axes(ax, grid_axis="x")
ys = range(len(ranked))
ax.barh(list(ys), ranked["coverage_share"], height=0.7, linewidth=0, zorder=3,
        color=[srp_common.STATUS[v] for v in ranked["verdict"]])
ax.set_yticks(list(ys))
ax.set_yticklabels(ranked["metric"])
ax.invert_yaxis()
ax.axvline(COVERAGE_FLOOR, color=INK_MUTED, linewidth=1, linestyle="--", zorder=4)
ax.set_xlim(0, 1)
ax.set_xlabel("share of the cohort for which the metric is computable",
              fontsize=10, color=INK_MUTED)
ax.set_title("Candidate recovery metrics by coverage of the cohort",
             fontsize=13, fontweight="600", color=INK, loc="left", pad=10)
for y, (share, verdict) in enumerate(zip(ranked["coverage_share"], ranked["verdict"])):
    # The status colour is never the only cue -- every bar carries its verdict as text.
    ax.text(share + 0.012, y, f"{share:.0%}  {srp_common.verdict_label(verdict)}",
            va="center", fontsize=9, color=INK_MUTED)

cfg.save_figure(fig, "coverage_ranking", suppress=True,
                note=f"coverage floor {COVERAGE_FLOOR:.0%}")

# 3-agreement

### Do the surviving metrics measure the same thing?

Correlations are computed inside BigQuery over patients who have every metric, so only the
coefficients cross into the notebook. Near-redundant metrics collapse to whichever has better
coverage; near-orthogonal ones mean the project is choosing between different questions, not
different measurements of one.

In [ ]:
correlation_sql = CTES + """
    SELECT
        COUNT(*)                                         AS n_complete,
        CORR(postop_los_days, steps_recovery_ratio)      AS r_los_steps,
        CORR(postop_los_days, readmitted_30)             AS r_los_readmit,
        CORR(steps_recovery_ratio, readmitted_30)        AS r_steps_readmit
    FROM metrics
    WHERE steps_recovery_ratio IS NOT NULL
      AND readmitted_30 IS NOT NULL
"""

correlations = cached_gbq(correlation_sql, "metric_correlations").iloc[0]
n_complete = int(correlations["n_complete"])

if n_complete < SUPPRESS_THRESHOLD:
    print(f"Only {n_complete} patients have all three metrics, below the "
          f"{SUPPRESS_THRESHOLD}-patient threshold. Do not report these coefficients.")
else:
    print(f"patients with all three metrics: {n_complete:,}\n")
    for label, key in [("post-op LOS         vs  steps recovery ratio", "r_los_steps"),
                       ("post-op LOS         vs  30-day readmission", "r_los_readmit"),
                       ("steps recovery ratio vs  30-day readmission", "r_steps_readmit")]:
        r = correlations[key]
        reading = ("near-redundant" if abs(r) >= 0.7 else
                   "related" if abs(r) >= 0.4 else "largely independent")
        print(f"{label:44s} r = {r:+.3f}   ({reading})")
    print()
    print("Correlations against readmission are point-biserial -- it is binary, so the "
          "coefficient is bounded well below 1 and is not comparable to the LOS/steps one.")

### Where do the metrics disagree about the same patient?

The cross-tabulation of tertiles is the most informative single table in the audit: the
off-diagonal cells are patients who recover well by one measure and badly by another. Any cell
below the threshold is aggregated before the table is shown.

In [ ]:
crosstab_sql = CTES + """
    , tertiles AS (
        SELECT
            NTILE(3) OVER (ORDER BY steps_recovery_ratio) AS steps_tertile,
            NTILE(3) OVER (ORDER BY postop_los_days DESC) AS los_tertile
        FROM metrics
        WHERE steps_recovery_ratio IS NOT NULL
          AND postop_los_days IS NOT NULL
    )
    SELECT steps_tertile, los_tertile, COUNT(*) AS n_patients
    FROM tertiles
    GROUP BY steps_tertile, los_tertile
    ORDER BY steps_tertile, los_tertile
"""

crosstab = cached_gbq(crosstab_sql, "metric_discordance")
smallest = srp_common.min_cell(crosstab, "n_patients")

print(f"smallest cell: {smallest:,}")
if smallest < SUPPRESS_THRESHOLD:
    print(f"At least one cell is below the {SUPPRESS_THRESHOLD}-patient threshold, so this "
          "table may not be reported as-is. Widen the tertiles to halves, or aggregate.")
else:
    table = crosstab.pivot(index="steps_tertile", columns="los_tertile",
                           values="n_patients")
    print()
    print("rows = steps recovery tertile, columns = post-op LOS tertile; 1 = worst in both, "
          "since LOS is ordered descending so that a long stay sorts as a poor outcome")
    print(table.to_string())
    on_diagonal = sum(int(table.loc[i, i]) for i in table.index if i in table.columns)
    print(f"\nagreeing on tertile: {on_diagonal:,} of {int(crosstab['n_patients'].sum()):,} "
          f"({on_diagonal / crosstab['n_patients'].sum():.1%})")

# 4-decision

### Which metric becomes the primary, and which become the pre-declared secondaries?

The choice is recorded here in code so it is committed before any model is fitted, which is what
stops the best-performing outcome from being selected after the fact. Edit these three constants
once the sections above have run; the printed summary is the thing to paste into the writeup.

In [ ]:
# Fill these in from the evidence above. They are deliberately left unset so that an
# unedited notebook cannot be mistaken for a decision that was actually made.
PRIMARY_OUTCOME = None
SECONDARY_OUTCOMES = []
DECISION_RATIONALE = ""

if PRIMARY_OUTCOME is None:
    print("No primary outcome recorded yet. Run the sections above, then set "
          "PRIMARY_OUTCOME, SECONDARY_OUTCOMES and DECISION_RATIONALE in this cell.")
else:
    chosen = scorecard.set_index("metric")
    print(f"PRIMARY   : {PRIMARY_OUTCOME}")
    print(f"            coverage {chosen.loc[PRIMARY_OUTCOME, 'coverage_share']:.1%}  "
          f"type {chosen.loc[PRIMARY_OUTCOME, 'outcome_type']}  "
          f"leakage risk {chosen.loc[PRIMARY_OUTCOME, 'leakage_risk']}")
    for name in SECONDARY_OUTCOMES:
        print(f"SECONDARY : {name}  "
              f"(coverage {chosen.loc[name, 'coverage_share']:.1%})")
    print(f"\nRationale : {DECISION_RATIONALE}")
    print(f"\nDecided against CDR {cfg.CDR}, "
          f"index rule {INDEX_RULE}.")